In [1]:
import pandas as pd
import pulp


In [ ]:
# ----------------------------------------------------------------------
# 1. Load data (you may have them as CSV, Excel, etc.)
# ----------------------------------------------------------------------
# stops:   id, fixed_cost (F_s), km_cost (K_s), route_len (L_s)
stops_df = pd.read_csv("stops.csv")               # columns: id, F, K, L
# destinations (demand zones):  id, demand (w_d)
dest_df   = pd.read_csv("destinations.csv")       # columns: id, demand
# travel_time matrix: rows = demand ids, cols = stop ids, values = c_{d,s}
travel_df = pd.read_csv("travel_time.csv", index_col=0)   # matrix format

In [ ]:
# usant dades gtfs moventis
FREQ =  4.4
TIME_OUTSIDE_CITY = 45
TIME_REST = 5.833333333333333

In [ ]:
# ----------------------------------------------------------------------
# 2. Extract sets & parameters
# ----------------------------------------------------------------------
S = list(stops_df["id"])
D = list(dest_df["id"])

# demand per zone
demand = dict(zip(dest_df["id"], dest_df["demand"]))

# travel time (or generalized user cost) per (d,s) pair
travel_time = travel_df.to_dict()   # travel_time[d][s]

# stop‑specific parameters
fixed_cost   = dict(zip(stops_df["id"], stops_df["F"]))   # F_s
km_cost      = dict(zip(stops_df["id"], stops_df["K"]))   # K_s
route_len    = dict(zip(stops_df["id"], stops_df["L"]))   # L_s

# ----------------------------------------------------------------------
# 3. Global model parameters (tune these for your case)
# ----------------------------------------------------------------------
VOT          = 12.0          # €/hour  (example)
bus_capacity = 80            # passengers per vehicle
f_min        = 2.0           # vehicles per hour (min)
f_max        = 12.0          # vehicles per hour (max)
stop_budget  = None          # e.g. 15 → max number of stops; set to None to omit
# ----------------------------------------------------------------------


In [ ]:
# ----------------------------------------------------------------------
# 4. Initialise the problem
# ----------------------------------------------------------------------
prob = pulp.LpProblem("Interurban_Bus_Network_Design", pulp.LpMinimize)

# ----------------------------------------------------------------------
# 5. Decision variables
# ----------------------------------------------------------------------
# y_s : binary – is stop s opened?
y = pulp.LpVariable.dicts("OpenStop", S, lowBound=0, upBound=1, cat="Binary")

# x_{d,s} : binary – is demand d assigned to stop s?
x = pulp.LpVariable.dicts(
        "Assign",
        [(d, s) for d in D for s in S],
        lowBound=0, upBound=1, cat="Binary")

# f : continuous service frequency (vehicles per hour)
f = pulp.LpVariable("Frequency", lowBound=f_min, upBound=f_max, cat="Continuous")

# ----------------------------------------------------------------------
# 6. Objective = Operator cost (Co) + User cost (Cu)
# ----------------------------------------------------------------------
# Operator cost (paper’s Co):
#   Σ_s F_s·y_s               – fixed stop cost
# + Σ_s K_s·L_s·f·y_s          – variable km cost (proportional to frequency)
operator_cost = (
    pulp.lpSum(fixed_cost[s] * y[s]                     for s in S) +
    pulp.lpSum(km_cost[s] * route_len[s] * f * y[s]      for s in S)
)

# User cost (paper’s Cu):
#   VOT * [ Σ_{d,s} w_d·c_{d,s}·x_{d,s}          – in‑vehicle travel time
#         + (1/(2·f))· Σ_d w_d ]                – average waiting time (half headway)
waiting_time_term = (1.0 / (2.0 * f)) * pulp.lpSum(demand[d] for d in D)
travel_time_term   = pulp.lpSum(demand[d] * travel_time[d][s] * x[(d, s)]
                               for d in D for s in S)
user_cost = VOT * (travel_time_term + waiting_time_term)

# Complete objective
prob += operator_cost + user_cost, "Total_System_Cost"

# ----------------------------------------------------------------------
# 7. Constraints
# ----------------------------------------------------------------------
# 7.1 Assignment: each demand zone assigned to exactly one stop
for d in D:
    prob += pulp.lpSum(x[(d, s)] for s in S) == 1, f"AssignOnce_{d}"

# 7.2 Link: assignment only to opened stops
for d in D:
    for s in S:
        prob += x[(d, s)] <= y[s], f"Link_{d}_{s}"

# 7.3 Capacity constraint (single line, r = 1)
# Total passengers that must be carried ≤ frequency × vehicle capacity
prob += pulp.lpSum(demand[d] * x[(d, s)] for d in D for s in S) \
        <= f * bus_capacity, "Capacity"

# 7.4 (Optional) Stop‑budget: limit the number of opened stops
if stop_budget is not None:
    prob += pulp.lpSum(y[s] for s in S) <= stop_budget, "StopBudget"

# ----------------------------------------------------------------------
# 8. Solve
# ----------------------------------------------------------------------
# By default PuLP uses the CBC solver (free).  If you have Gurobi/CPLEX installed,
# replace the solver call, e.g. prob.solve(pulp.GUROBI_CMD())
prob.solve(pulp.PULP_CBC_CMD(msg=True))

# ----------------------------------------------------------------------
# 9. Extract results
# ----------------------------------------------------------------------
opened_stops = [s for s in S if pulp.value(y[s]) > 0.5]
assignments = {(d, s): pulp.value(x[(d, s)]) for d in D for s in S
               if pulp.value(x[(d, s)]) > 0.5}
frequency   = pulp.value(f)
total_cost  = pulp.value(prob.objective)
operator    = pulp.value(operator_cost)
user        = pulp.value(user_cost)

print("\n=== OPTIMAL SOLUTION ===")
print(f"Frequency (veh/h)      : {frequency:.2f}")
print(f"Opened stops ({len(opened_stops)}): {opened_stops}")
print(f"Total system cost (€)   : {total_cost:,.2f}")
print(f"  → Operator cost (€)   : {operator:,.2f}")
print(f"  → User cost (€)       : {user:,.2f}")
